# Week 1-4 · 선형 StateGraph를 직접 만들기

## 시나리오
정책 질문이 `retrieve → answer` node를 순서대로 지나도록 state, node, edge를 직접 정의하고 graph를 invoke합니다.

## 학습 목표
- `TypedDict`로 graph state를 선언한다.
- node가 자신이 담당하는 state 일부만 반환하도록 만든다.
- `START`, `END`, `add_node`, `add_edge`, `compile`, `invoke`를 사용한다.

## 직접 조립
완성된 `weekX.app` 함수를 가져오지 않습니다. 아래 코드에서 작은 fixture와 핵심 객체·함수·연결을 직접 만듭니다.

### 1단계 · state와 node 정의

In [ ]:
# 실행 순서: 1단계 · state와 node 정의에서 PracticeState, retrieve_node, answer_node을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 1단계 · state와 node 정의.
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

# 각 node가 이어받을 질문·근거·답변·실행 trace를 한 state에 선언합니다.
class PracticeState(TypedDict):
    question: str
    evidence: list[str]
    answer: str
    steps: list[str]

practice_policy = "휴가는 시작일 3영업일 전에 신청합니다."

# 질문과 맞는 fixture 근거만 state의 evidence에 추가합니다.
def retrieve_node(state: PracticeState) -> dict:
    evidence = [practice_policy] if "휴가" in state["question"] else []
    return {"evidence": evidence, "steps": state["steps"] + ["retrieve"]}

# 앞 node가 남긴 evidence만 사용해 답변하며 근거가 없으면 생성하지 않습니다.
def answer_node(state: PracticeState) -> dict:
    answer = state["evidence"][0] if state["evidence"] else "근거 없음"
    return {"answer": answer, "steps": state["steps"] + ["answer"]}

### 2단계 · node와 edge 연결

In [ ]:
# 실행 순서: 2단계 · node와 edge 연결에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 2단계 · node와 edge 연결.
practice_builder = StateGraph(PracticeState)
practice_builder.add_node("retrieve", retrieve_node)
practice_builder.add_node("answer", answer_node)
practice_builder.add_edge(START, "retrieve")
practice_builder.add_edge("retrieve", "answer")
practice_builder.add_edge("answer", END)
practice_graph = practice_builder.compile()

### 3단계 · graph invoke와 state 관찰

In [ ]:
# 실행 순서: 3단계 · graph invoke와 state 관찰에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 3단계 · graph invoke와 state 관찰.
practice_result = practice_graph.invoke({
    "question": "휴가는 언제 신청하나요?",
    "evidence": [],
    "answer": "",
    "steps": [],
})
assert practice_result["steps"] == ["retrieve", "answer"]
assert practice_result["evidence"] == [practice_policy]
assert "3영업일" in practice_result["answer"]
practice_result

## 중간 결과
각 코드 셀의 출력에서 입력이 어떤 상태로 변했는지 확인합니다. 마지막 `assert`는 눈으로 본 결과를 실행 가능한 계약으로 고정합니다.

## 실패 경계
node가 이전 state를 임의로 지우지 않아야 합니다. 근거가 없을 때 답을 만드는 문제는 다음 Notebook의 evidence gate에서 다룹니다.

## 실제 app 연결
Week 1 app은 이 선형 graph에 실제 retriever와 grounding guard를 추가합니다. 이 실습은 그 전 단계인 node 생성·연결·state 전달을 최소 예제로 분리했습니다.

### 확장 과제
fixture의 문장이나 임계값을 하나 바꾸고, 어느 중간 결과와 assertion이 달라지는지 기록하세요.

## 다음 Notebook 연결
다음 `05_grounded_api_testing.ipynb`에서는 graph에서 배운 단계 흐름에 evidence gate와 FastAPI 경계를 결합합니다.